# LeetCode 805 — Split Array With Same Average bằng Python

- Link: https://leetcode.com/problems/split-array-with-same-average/
- Độ khó: **Hard**
- Chủ đề: Toán (trung bình), Quy hoạch động tổng tập con theo số phần tử, Bitset, Meet in the middle

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def splitArraySameAverage(self, nums: List[int]) -> bool:
```

## 1. Giới thiệu & mục tiêu học

Chia mảng `nums` thành hai mảng **khác rỗng** `A` và `B` sao cho **trung bình cộng** bằng nhau. Có làm được không?

Với `n ≤ 30`, thử mọi cách chia là `2³⁰ ≈ 10⁹` — quá nhiều. Bài này là bài tập tuyệt vời về **biến đổi toán học** trước khi code: "trung bình A = trung bình B" tương đương "trung bình A = trung bình cả mảng", tương đương "tồn tại tập con `k` phần tử có tổng **đúng bằng** `k · S / n`". Từ đó có hai hướng: **DP tổng theo số phần tử** và **meet in the middle**.

Sau notebook này bạn sẽ:

1. Chứng minh `avg(A) = avg(B) ⇔ avg(A) = avg(nums)`.
2. Quy về bài **tổng tập con có ràng buộc số phần tử** và dùng điều kiện chia hết để cắt tỉa.
3. Cài DP `dp[k] = tập tổng của các tập con k phần tử`, rồi tăng tốc bằng **bitset** (số nguyên Python).
4. Cài **meet in the middle** với phép biến đổi `aᵢ = n·nums[i] − S` (tìm tập con tổng 0).
5. Tránh các bẫy: tập con rỗng, tập con là cả mảng, chỉ cần xét `k ≤ n/2`.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho mảng số nguyên `nums`. Ta phải đưa **mỗi** phần tử vào đúng một trong hai mảng `A` hoặc `B`, sao cho `A` và `B` đều **khác rỗng** và `average(A) == average(B)`. Trả về `True` nếu làm được.

`average(arr)` = tổng các phần tử chia cho số phần tử.

**Ví dụ 1**

```text
Input: nums = [1,2,3,4,5,6,7,8]
Output: true
Giải thích: A = [1,4,5,8], B = [2,3,6,7], cả hai có trung bình 4.5.
```

**Ví dụ 2**

```text
Input: nums = [3,1]
Output: false
```

**Ràng buộc**

- `1 <= nums.length <= 30`
- `0 <= nums[i] <= 10⁴`

In [1]:
from typing import List

EXAMPLES = [
    ([1, 2, 3, 4, 5, 6, 7, 8], True),
    ([3, 1], False),
    ([5], False),
    ([0, 0], True),
    ([2, 0, 5, 6, 16, 12, 15, 12, 4], True),
    ([18, 10, 5, 3], False),
]
for a, exp in EXAMPLES:
    print(a, "->", exp, "| trung bình cả mảng =", sum(a) / len(a))

[1, 2, 3, 4, 5, 6, 7, 8] -> True | trung bình cả mảng = 4.5
[3, 1] -> False | trung bình cả mảng = 2.0
[5] -> False | trung bình cả mảng = 5.0
[0, 0] -> True | trung bình cả mảng = 0.0
[2, 0, 5, 6, 16, 12, 15, 12, 4] -> True | trung bình cả mảng = 8.0
[18, 10, 5, 3] -> False | trung bình cả mảng = 9.0


## 3. Kiến thức nền

### 3.1. Tránh số thực: so sánh trung bình bằng nhân chéo

`sA/kA == sB/kB` ⇔ `sA·kB == sB·kA` (với `kA, kB > 0`). Luôn dùng số nguyên để tránh sai số dấu phẩy động.

In [2]:
print(0.1 + 0.2 == 0.3, (1 * 10 + 2 * 10) == 3 * 10)
sA, kA, sB, kB = 18, 4, 18, 4
print(sA * kB == sB * kA)

False True
True


### 3.2. Duyệt tập con bằng bitmask và `combinations`

`itertools.combinations(nums, k)` sinh mọi bộ `k` phần tử (theo vị trí). Tổng số tập con của `n` phần tử là `2ⁿ`.

In [3]:
from itertools import combinations
print(list(combinations([1, 2, 3, 4], 2)))
print(sum(1 for k in range(5) for _ in combinations(range(4), k)), "= 2^4")

[(1, 2), (1, 3), (1, 4), (2, 3), (2, 4), (3, 4)]
16 = 2^4


### 3.3. Tập hợp (`set`) cho DP tổng tập con

`sums = {0}`; với mỗi `x`: `sums |= {s + x for s in sums}`. Sau khi duyệt hết, `sums` chứa mọi tổng tập con.

In [4]:
sums = {0}
for x in [3, 5, 9]:
    sums |= {s + x for s in sums}
print(sorted(sums))

[0, 3, 5, 8, 9, 12, 14, 17]


### 3.4. Bitset bằng số nguyên Python

Biểu diễn tập tổng bằng một số nguyên `B`: bit `s` bật ⇔ tổng `s` đạt được. Thêm phần tử `x`: `B |= B << x`. Python hỗ trợ số nguyên lớn tuỳ ý, và phép dịch/OR chạy ở tốc độ C → rất nhanh.

In [5]:
B = 1                      # chỉ tổng 0
for x in [3, 5, 9]:
    B |= B << x
print([s for s in range(B.bit_length()) if B >> s & 1])

[0, 3, 5, 8, 9, 12, 14, 17]


### 3.5. Meet in the middle

Chia mảng làm hai nửa, mỗi nửa ~15 phần tử → `2¹⁵ = 32768` tập con mỗi nửa. Liệt kê tổng ở mỗi nửa rồi **ghép** hai nửa bằng tra cứu `set` thay vì thử `2³⁰` tổ hợp.

In [6]:
def subset_sums(arr):
    """Danh sách (tổng, mask) của mọi tập con."""
    res = [(0, 0)]
    for i, x in enumerate(arr):
        res += [(s + x, m | (1 << i)) for s, m in res]
    return res

print(subset_sums([1, 2, 4]))

[(0, 0), (1, 1), (2, 2), (3, 3), (4, 4), (5, 5), (6, 6), (7, 7)]


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Bổ đề then chốt

Gọi `S = sum(nums)`, `n = len(nums)`; `A` có `k` phần tử, tổng `sA`; `B` có `n − k` phần tử, tổng `S − sA`.

```text
sA / k = (S − sA) / (n − k)
⇔ sA·(n − k) = (S − sA)·k
⇔ sA·n − sA·k = S·k − sA·k
⇔ sA·n = S·k
⇔ sA / k = S / n
```

→ **`avg(A) = avg(B)` ⇔ `avg(A) = avg(nums)`** ⇔ `sA = S·k / n`.

### 4.2. Hệ quả

1. Cần tồn tại `k ∈ [1, n−1]` và tập con `k` phần tử có tổng **đúng** `S·k/n` — và `S·k` phải **chia hết** cho `n`.
2. Nếu `A` thoả thì `B` cũng thoả (cùng trung bình). Vậy chỉ cần xét `k ≤ n/2` (lấy tập nhỏ hơn).

### 4.3. Tính tay ví dụ 1

`nums = [1..8]`, `S = 36`, `n = 8`, `S/n = 4.5`.

| k | S·k | chia hết cho 8? | tổng cần |
|---|---|---|---|
| 1 | 36 | ✘ | — |
| 2 | 72 | ✔ | 9 → ví dụ `{1, 8}` ✔ |

Vậy `A = [1, 8]`, `B = [2..7]`: trung bình đều 4.5 → `True`.

### 4.4. Tính tay ví dụ 2

`nums = [3, 1]`, `S = 4`, `n = 2`. `k = 1`: `S·k = 4`, chia hết cho 2, tổng cần 2. Tập con 1 phần tử có tổng 2? Không (`3` hoặc `1`) → `False`.

In [7]:
def needed_sums(nums):
    S, n = sum(nums), len(nums)
    return [(k, S * k // n) for k in range(1, n // 2 + 1) if S * k % n == 0]

print(needed_sums([1, 2, 3, 4, 5, 6, 7, 8]))
print(needed_sums([3, 1]))
print(needed_sums([18, 10, 5, 3]))

[(2, 9), (4, 18)]
[(1, 2)]
[(1, 9), (2, 18)]


## 5. Cách 1 — Brute Force: thử mọi cách chia

**Ý tưởng.** Duyệt mọi `mask` từ `1` tới `2ⁿ − 2` (bỏ rỗng và cả mảng). `A` = các phần tử có bit bật. Kiểm tra `sA·(n − k) == (S − sA)·k`.

In [8]:
def split_brute(nums):
    n, S = len(nums), sum(nums)
    for mask in range(1, (1 << n) - 1):
        k = bin(mask).count('1')
        sA = sum(nums[i] for i in range(n) if mask >> i & 1)
        if sA * (n - k) == (S - sA) * k:
            return True
    return False

for a, exp in EXAMPLES:
    print(a, split_brute(a), "(kỳ vọng", exp, ")")
    assert split_brute(a) == exp

[1, 2, 3, 4, 5, 6, 7, 8] True (kỳ vọng True )
[3, 1] False (kỳ vọng False )
[5] False (kỳ vọng False )
[0, 0] True (kỳ vọng True )
[2, 0, 5, 6, 16, 12, 15, 12, 4] True (kỳ vọng True )
[18, 10, 5, 3] False (kỳ vọng False )


**Độ phức tạp.** `O(2ⁿ · n)`. Với `n = 30`: ~`3·10¹⁰` → bất khả thi. Chỉ dùng cho `n ≤ 16` để đối chiếu.

**Nút thắt.** Nhiều tập con có **cùng (số phần tử, tổng)** — ta chỉ cần biết cặp `(k, tổng)` nào **đạt được**, không cần biết tập con cụ thể. Số cặp `(k, tổng)` tối đa `30 × 3·10⁵` — nhỏ hơn `2³⁰` rất nhiều → DP.

## 6. Bước đột phá

**Insight 1 — Quy về trung bình cả mảng** (mục 4.1): cần tập con `k` phần tử tổng `S·k/n`, `1 ≤ k ≤ n/2`.

**Insight 2 — Cắt tỉa nhanh.** *Nếu* không có `k ∈ [1, n/2]` nào với `S·k % n == 0` *thì* trả `False` ngay.

**Insight 3 — DP theo (số phần tử, tổng).** `dp[k]` = tập tổng đạt được bởi tập con **đúng `k`** phần tử. Thêm phần tử `x`: duyệt `k` **giảm dần** (giống knapsack 0/1) `dp[k] |= {s + x for s in dp[k−1]}`.

**Insight 4 — Bitset.** `dp[k]` là một số nguyên; `dp[k] |= dp[k−1] << x`.

**Insight 5 — Meet in the middle với biến đổi tổng 0.** Đặt `aᵢ = n·nums[i] − S`. Khi đó `avg(A) = S/n` ⇔ `Σ_{i∈A} aᵢ = n·sA − k·S = 0`. Bài toán thành: **tìm tập con khác rỗng, khác cả mảng, có tổng 0** (không còn phụ thuộc `k`!). Chia đôi, ghép bằng `set`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DP tập tổng theo số phần tử (`set`)

In [9]:
def split_dp_set(nums):
    n, S = len(nums), sum(nums)
    half = n // 2
    if not any(S * k % n == 0 for k in range(1, half + 1)):
        return False
    dp = [set() for _ in range(half + 1)]
    dp[0].add(0)
    for x in nums:
        for k in range(half, 0, -1):          # giảm dần: mỗi phần tử dùng tối đa 1 lần
            if dp[k - 1]:
                dp[k] |= {s + x for s in dp[k - 1]}
    return any(S * k % n == 0 and S * k // n in dp[k] for k in range(1, half + 1))

for a, exp in EXAMPLES:
    assert split_dp_set(a) == exp
print("DP set: OK")

DP set: OK


Code nộp LeetCode cho cách 2:

In [10]:
class SolutionDPSet:
    def splitArraySameAverage(self, nums: List[int]) -> bool:
        n, S = len(nums), sum(nums)
        half = n // 2
        if not any(S * k % n == 0 for k in range(1, half + 1)):
            return False
        dp = [set() for _ in range(half + 1)]
        dp[0].add(0)
        for x in nums:
            for k in range(half, 0, -1):
                if dp[k - 1]:
                    dp[k].update(s + x for s in dp[k - 1])
        return any(S * k % n == 0 and S * k // n in dp[k] for k in range(1, half + 1))

for a, exp in EXAMPLES:
    assert SolutionDPSet().splitArraySameAverage(a) == exp
print("SolutionDPSet: OK")

SolutionDPSet: OK


**Độ phức tạp cách 2:** `n` phần tử × `n/2` giá trị `k` × `|dp[k−1]|` (≤ `S`) → `O(n² · S)` trường hợp xấu (`30² · 3·10⁵ ≈ 2.7·10⁸` thao tác set — chậm trong Python nếu gặp test xấu, nhưng thực tế các tập thường nhỏ hơn nhiều và cắt tỉa giúp bỏ qua nhiều test). Bộ nhớ `O(n · S)`.

Một tối ưu hay dùng: **sắp xếp** `nums` và chỉ giữ các tổng `≤ S·k/n` cần thiết... ở đây ta chuyển thẳng sang bitset.

### 7.2. Cách 3 — DP bitset theo số phần tử

In [11]:
def split_bitset(nums):
    n, S = len(nums), sum(nums)
    half = n // 2
    if not any(S * k % n == 0 for k in range(1, half + 1)):
        return False
    dp = [0] * (half + 1)
    dp[0] = 1                                  # bit 0: tổng 0 với 0 phần tử
    for x in nums:
        for k in range(half, 0, -1):
            dp[k] |= dp[k - 1] << x
    return any(S * k % n == 0 and dp[k] >> (S * k // n) & 1 for k in range(1, half + 1))

for a, exp in EXAMPLES:
    assert split_bitset(a) == exp
print("DP bitset: OK")

DP bitset: OK


In [12]:
class SolutionBitset:
    def splitArraySameAverage(self, nums: List[int]) -> bool:
        n, S = len(nums), sum(nums)
        half = n // 2
        targets = [k for k in range(1, half + 1) if S * k % n == 0]
        if not targets:
            return False
        dp = [0] * (half + 1)
        dp[0] = 1
        for x in nums:
            for k in range(half, 0, -1):
                dp[k] |= dp[k - 1] << x
        return any(dp[k] >> (S * k // n) & 1 for k in targets)

for a, exp in EXAMPLES:
    assert SolutionBitset().splitArraySameAverage(a) == exp
print("SolutionBitset: OK")

SolutionBitset: OK


**Độ phức tạp cách 3:** `n · n/2` phép dịch/OR trên số `S` bit → `O(n² · S / w)` với `w` là độ rộng từ máy (64). ≈ `450 × 3·10⁵/64 ≈ 2·10⁶` phép từ — rất nhanh.

### 7.3. Cách 4 — Meet in the middle (tổng 0)

1. `a[i] = n·nums[i] − S` (tổng toàn bộ `a` bằng 0).
2. Chia `a` thành `L` (nửa đầu) và `R` (nửa sau).
3. Liệt kê tổng các tập con **khác rỗng** của `L`, của `R`.
4. Nếu có tập con khác rỗng của `L` (hoặc `R`) tổng 0 → `True` (nó không thể là cả mảng vì nửa kia khác rỗng — khi `n ≥ 2`).
5. Ghép: tồn tại `sL` (tập con khác rỗng, **khác toàn bộ `L`**) và `sR` (tập con khác rỗng của `R`) với `sL + sR = 0` → `True`.

Vì sao loại "toàn bộ `L`" ở bước 5? Để tránh chọn `L ∪ R` = cả mảng. Trường hợp `A = L ∪ (một phần của R)` không bị bỏ sót: phần bù của nó là `R \ phần đó` — một tập con khác rỗng của `R` có tổng 0, đã được bước 4 phát hiện.

In [13]:
def split_mitm(nums):
    n, S = len(nums), sum(nums)
    if n == 1:
        return False
    a = [n * x - S for x in nums]
    L, R = a[: n // 2], a[n // 2:]

    def sums_nonempty(arr, exclude_full):
        full = (1 << len(arr)) - 1
        res = set()
        cur = [(0, 0)]
        for i, x in enumerate(arr):
            cur += [(s + x, m | (1 << i)) for s, m in cur]
        for s, m in cur:
            if m and not (exclude_full and m == full):
                res.add(s)
        return res

    left = sums_nonempty(L, exclude_full=False)
    right = sums_nonempty(R, exclude_full=False)
    if 0 in left or 0 in right:
        return True
    left_partial = sums_nonempty(L, exclude_full=True)
    return any(-s in right for s in left_partial)

for a, exp in EXAMPLES:
    assert split_mitm(a) == exp, a
print("Meet in the middle: OK")

Meet in the middle: OK


Code nộp LeetCode cho cách 4 (sinh tổng tập con bằng cách nhân đôi danh sách, mỗi nửa ≤ 15 phần tử → ≤ 32768 tổng):

In [14]:
class SolutionMITM:
    def splitArraySameAverage(self, nums: List[int]) -> bool:
        n, S = len(nums), sum(nums)
        if n == 1:
            return False
        a = [n * x - S for x in nums]
        L, R = a[: n // 2], a[n // 2:]

        def gen(arr):
            sums = [0]
            for x in arr:
                sums += [s + x for s in sums]
            return sums                     # sums[mask] = tổng tập con mask

        sl, sr = gen(L), gen(R)
        fullL = len(sl) - 1
        right = set(sr[1:])                 # bỏ tập rỗng
        if 0 in right or 0 in sl[1:]:
            return True
        return any(-sl[m] in right for m in range(1, fullL))

for a, exp in EXAMPLES:
    assert SolutionMITM().splitArraySameAverage(a) == exp
print("SolutionMITM: OK")

SolutionMITM: OK


Mẹo trong `gen`: khi nhân đôi danh sách `sums += [s + x for s in sums]`, phần tử ở vị trí `mask` đúng bằng tổng tập con có bitmask `mask` (bit `i` ứng với phần tử thứ `i`). Nên `sl[fullL]` là tổng cả `L`, và `range(1, fullL)` duyệt mọi tập con khác rỗng, khác toàn bộ.

**Độ phức tạp cách 4:** `O(2^{n/2})` thời gian và bộ nhớ ≈ `3.3·10⁴` mỗi nửa — không phụ thuộc độ lớn giá trị!

## 8. TRACE từng bước

### 8.1. Trace cách 2 (DP set) trên `nums = [1, 2, 3, 4, 5, 6, 7, 8]`

In [15]:
def trace_dp_set(nums):
    n, S = len(nums), sum(nums)
    half = n // 2
    targets = {k: S * k // n for k in range(1, half + 1) if S * k % n == 0}
    print(f"S={S}, n={n}, các k hợp lệ và tổng cần: {targets}")
    dp = [set() for _ in range(half + 1)]
    dp[0].add(0)
    for idx, x in enumerate(nums):
        for k in range(half, 0, -1):
            if dp[k - 1]:
                dp[k] |= {s + x for s in dp[k - 1]}
        info = ", ".join(f"k={k}:{sorted(dp[k])[:6]}{'...' if len(dp[k]) > 6 else ''}" for k in range(1, min(3, half) + 1))
        hit = [k for k, t in targets.items() if t in dp[k]]
        print(f"sau x={x}: {info} | đạt mục tiêu ở k = {hit}")
    return any(t in dp[k] for k, t in targets.items())

print("=>", trace_dp_set([1, 2, 3, 4, 5, 6, 7, 8]))

S=36, n=8, các k hợp lệ và tổng cần: {2: 9, 4: 18}
sau x=1: k=1:[1], k=2:[], k=3:[] | đạt mục tiêu ở k = []
sau x=2: k=1:[1, 2], k=2:[3], k=3:[] | đạt mục tiêu ở k = []
sau x=3: k=1:[1, 2, 3], k=2:[3, 4, 5], k=3:[6] | đạt mục tiêu ở k = []
sau x=4: k=1:[1, 2, 3, 4], k=2:[3, 4, 5, 6, 7], k=3:[6, 7, 8, 9] | đạt mục tiêu ở k = []
sau x=5: k=1:[1, 2, 3, 4, 5], k=2:[3, 4, 5, 6, 7, 8]..., k=3:[6, 7, 8, 9, 10, 11]... | đạt mục tiêu ở k = [2]
sau x=6: k=1:[1, 2, 3, 4, 5, 6], k=2:[3, 4, 5, 6, 7, 8]..., k=3:[6, 7, 8, 9, 10, 11]... | đạt mục tiêu ở k = [2, 4]
sau x=7: k=1:[1, 2, 3, 4, 5, 6]..., k=2:[3, 4, 5, 6, 7, 8]..., k=3:[6, 7, 8, 9, 10, 11]... | đạt mục tiêu ở k = [2, 4]
sau x=8: k=1:[1, 2, 3, 4, 5, 6]..., k=2:[3, 4, 5, 6, 7, 8]..., k=3:[6, 7, 8, 9, 10, 11]... | đạt mục tiêu ở k = [2, 4]
=> True


Bảng tóm tắt (ví dụ 1): mục tiêu `k=2 → 9`, `k=4 → 18`. Sau khi thêm `x = 5`, `dp[2]` đã chứa `9` (`4 + 5`) → có thể dừng sớm (trong code nộp ta kiểm tra ở cuối cho đơn giản).

### 8.2. Trace cách 4 (meet in the middle) trên `nums = [3, 1]` và `[1, 2, 3, 4]`

In [16]:
def trace_mitm(nums):
    n, S = len(nums), sum(nums)
    a = [n * x - S for x in nums]
    L, R = a[: n // 2], a[n // 2:]
    print(f"nums={nums}, S={S}, a = n*x - S = {a}, L={L}, R={R}")
    def gen(arr):
        sums = [0]
        for x in arr:
            sums += [s + x for s in sums]
        return sums
    sl, sr = gen(L), gen(R)
    print("  tổng tập con L (theo mask):", sl)
    print("  tổng tập con R (theo mask):", sr)
    right = set(sr[1:])
    if 0 in right or 0 in sl[1:]:
        print("  có nửa nào đó chứa tập con tổng 0 -> True")
        return True
    for m in range(1, len(sl) - 1):
        if -sl[m] in right:
            print(f"  ghép: mask L={m:b} tổng {sl[m]} với tổng {-sl[m]} bên R -> True")
            return True
    print("  không ghép được -> False")
    return False

trace_mitm([3, 1])
trace_mitm([1, 2, 3, 4])

nums=[3, 1], S=4, a = n*x - S = [2, -2], L=[2], R=[-2]
  tổng tập con L (theo mask): [0, 2]
  tổng tập con R (theo mask): [0, -2]
  không ghép được -> False
nums=[1, 2, 3, 4], S=10, a = n*x - S = [-6, -2, 2, 6], L=[-6, -2], R=[2, 6]
  tổng tập con L (theo mask): [0, -6, -2, -8]
  tổng tập con R (theo mask): [0, 2, 6, 8]
  ghép: mask L=1 tổng -6 với tổng 6 bên R -> True


True

Với `[1, 2, 3, 4]`: `S = 10`, `n = 4`, `a = [−6, −2, 2, 6]`. `L = [−6, −2]`, `R = [2, 6]`. Tập con `{−6}` của `L` ghép với `{6}` của `R` → `A = {1, 4}` (trung bình 2.5), `B = {2, 3}` (trung bình 2.5). ✔

## 9. Code nộp LeetCode

Chọn cách 3 (bitset) — ngắn, nhanh, dễ nhớ:

In [17]:
class Solution:
    def splitArraySameAverage(self, nums: List[int]) -> bool:
        n, S = len(nums), sum(nums)
        half = n // 2
        # chỉ những k có S*k chia hết cho n mới có thể đạt trung bình S/n
        targets = [k for k in range(1, half + 1) if S * k % n == 0]
        if not targets:
            return False
        dp = [0] * (half + 1)      # dp[k]: bit s bật <=> có tập con k phần tử tổng s
        dp[0] = 1
        for x in nums:
            for k in range(half, 0, -1):
                dp[k] |= dp[k - 1] << x
        return any(dp[k] >> (S * k // n) & 1 for k in targets)

for a, exp in EXAMPLES:
    assert Solution().splitArraySameAverage(a) == exp
print("Solution: OK")

Solution: OK


## 10. Vì sao thuật toán đúng

**Bổ đề (mục 4.1).** Với `1 ≤ k ≤ n − 1`: `avg(A) = avg(B)` ⇔ `n·sA = k·S`.

**Chỉ cần `k ≤ n/2`.** Nếu `A` (k phần tử) thoả thì `B` (n−k phần tử) cũng có `avg(B) = S/n`, tức `B` cũng thoả. Một trong hai có kích thước `≤ n/2`.

**Cách 2/3.** Bất biến của DP 0/1: sau khi xử lý `t` phần tử đầu, `dp[k]` = tập tổng của các tập con `k` phần tử trong `t` phần tử đầu. Duyệt `k` giảm dần đảm bảo `dp[k−1]` dùng ở bước này là giá trị **trước** khi thêm phần tử hiện tại → mỗi phần tử dùng tối đa một lần. Cuối cùng kiểm tra đúng điều kiện của bổ đề.

**Cách 4.** `Σ_{i∈A} aᵢ = n·sA − k·S`. Theo bổ đề, `A` hợp lệ ⇔ `A` khác rỗng, khác cả mảng, `Σ_{A} a = 0`. Mọi `A` như vậy phân tách thành `A_L = A ∩ L`, `A_R = A ∩ R`:
- `A_R = ∅` → `A_L` khác rỗng tổng 0 → bước 4 bắt được.
- `A_L = ∅` → tương tự.
- Cả hai khác rỗng, `A_L ≠ L` → bước 5 bắt được.
- `A_L = L`, `A_R ≠ R` (vì `A` ≠ cả mảng) → phần bù `R \ A_R` khác rỗng, tổng `0 − 0 = 0` (vì `Σ a = 0` và `Σ_A a = 0`) → bước 4 bắt được.

Ngược lại mọi trường hợp thuật toán trả `True` đều ứng với một `A` hợp lệ. ∎

## 11. Phân tích Time/Space

| Cách | Time | Space |
|---|---|---|
| 1. Brute force | `O(2ⁿ · n)` | `O(1)` |
| 2. DP set | `O(n² · S)` (xấu nhất) | `O(n · S)` |
| 3. DP bitset | `O(n² · S / 64)` | `O(n · S / 64)` |
| 4. Meet in the middle | `O(2^{n/2} · n)` | `O(2^{n/2})` |

(`S ≤ 3·10⁵`.) Đo trên test xấu: 30 phần tử lớn, có lời giải và không có lời giải.

In [18]:
import random, time

rnd = random.Random(805)
hard_true = [rnd.randint(0, 10**4) for _ in range(29)]
hard_true.append((-sum(hard_true)) % 30)    # S chia hết cho 30 -> mọi k đều hợp lệ, DP phải làm đủ việc
hard_false = [10**4] * 29 + [1]          # S = 290001, chỉ k = 10 hợp lệ
for name, fn in [("dp_set", split_dp_set), ("bitset", split_bitset), ("mitm", split_mitm),
                 ("Solution", lambda a: Solution().splitArraySameAverage(a))]:
    for label, arr in [("S chia hết", hard_true), ("gần như đều", hard_false)]:
        t0 = time.perf_counter()
        r = fn(arr)
        print(f"{name:8} | {label:11}: {r!s:5} ({(time.perf_counter() - t0) * 1000:.0f} ms)")

dp_set   | S chia hết : True  (403 ms)
dp_set   | gần như đều: False (0 ms)
bitset   | S chia hết : True  (1 ms)
bitset   | gần như đều: False (1 ms)
mitm     | S chia hết : True  (14 ms)
mitm     | gần như đều: False (16 ms)
Solution | S chia hết : True  (1 ms)
Solution | gần như đều: False (1 ms)


## 12. Bộ test

In [19]:
methods = {
    "brute": split_brute,
    "dp_set": split_dp_set,
    "bitset": split_bitset,
    "mitm": split_mitm,
    "SolutionDPSet": lambda a: SolutionDPSet().splitArraySameAverage(a),
    "SolutionBitset": lambda a: SolutionBitset().splitArraySameAverage(a),
    "SolutionMITM": lambda a: SolutionMITM().splitArraySameAverage(a),
    "Solution": lambda a: Solution().splitArraySameAverage(a),
}
fast = [m for m in methods if m != "brute"]

# 12.1 Ví dụ + edge cases
edge = EXAMPLES + [
    ([0], False),
    ([7, 7], True),                    # hai phần tử bằng nhau
    ([1, 2], False),
    ([1, 2, 3], True),                 # {2} và {1,3}
    ([10000] * 30, True),
    ([0] * 29 + [1], False),
    ([6, 8, 18, 3, 1], False),
    ([60, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 30], False),
]
for a, exp in edge:
    if len(a) <= 16:
        assert split_brute(a) == exp, a
    for name in fast:
        assert methods[name](a) == exp, (name, a)
print("Ví dụ + edge cases: OK")

# 12.2 Ngẫu nhiên có seed, n nhỏ: đối chiếu brute force
cnt_true = 0
for seed in range(1500):
    r = random.Random(seed)
    n = r.randint(1, 12)
    a = [r.randint(0, r.choice([5, 20, 100])) for _ in range(n)]
    ref = split_brute(a)
    cnt_true += ref
    for name, fn in methods.items():
        assert fn(a) == ref, (name, a)
print("1500 mảng ngẫu nhiên: khớp brute force; số True =", cnt_true)

# 12.3 n lớn (tới 30): các cách nhanh khớp nhau
for seed in range(60):
    r = random.Random(7000 + seed)
    n = r.randint(17, 30)
    a = [r.randint(0, 10**4) for _ in range(n)]
    if seed % 3 == 0:                  # tạo test "khó" nhiều False: giá trị lẻ + một phần tử lạ
        a = [r.choice([9999, 10000]) for _ in range(n - 1)] + [r.randint(0, 10**4)]
    vals = {name: methods[name](a) for name in fast}
    assert len(set(vals.values())) == 1, (a, vals)
print("n lớn: các cách nhanh khớp nhau")

Ví dụ + edge cases: OK


1500 mảng ngẫu nhiên: khớp brute force; số True = 210


n lớn: các cách nhanh khớp nhau


## 13. Lỗi người mới hay gặp

### Lỗi 1 — So sánh trung bình bằng số thực

**Code sai:**

In [20]:
def wrong_float(nums):
    n, S = len(nums), sum(nums)
    avg = S / n
    for k in range(1, n // 2 + 1):
        for comb in combinations(nums, k):
            if sum(comb) / k == avg:          # SAI (tiềm ẩn): sai số dấu phẩy động
                return True
    return False

# Ví dụ: 0.1 không biểu diễn chính xác; với số nguyên thì thường ổn nhưng không có gì đảm bảo
print(1 / 3 * 3 == 1, 0.1 * 3 == 0.3)
print("wrong_float([1,2,3,4,5,6,7,8]) =", wrong_float([1, 2, 3, 4, 5, 6, 7, 8]))

True False
wrong_float([1,2,3,4,5,6,7,8]) = True


**Vì sao sai:** phép chia số thực có sai số làm tròn; hai trung bình bằng nhau về mặt toán học có thể khác nhau ở bit cuối (ví dụ `0.1*3 != 0.3`). Ngoài ra duyệt `combinations` vẫn là `O(2ⁿ)`.

**Code đúng:** so sánh `n·sA == k·S` bằng số nguyên.

### Lỗi 2 — Duyệt `k` tăng dần trong DP (dùng một phần tử nhiều lần)

**Code sai:**

In [21]:
def wrong_increasing_k(nums):
    n, S = len(nums), sum(nums)
    half = n // 2
    dp = [0] * (half + 1); dp[0] = 1
    for x in nums:
        for k in range(1, half + 1):           # SAI: tăng dần
            dp[k] |= dp[k - 1] << x
    return any(S * k % n == 0 and dp[k] >> (S * k // n) & 1 for k in range(1, half + 1))

for arr in ([5, 2, 8, 11, 8, 4], [1, 2, 3, 4, 5, 6, 7, 8]):
    print(arr, "sai:", wrong_increasing_k(arr), " đúng:", Solution().splitArraySameAverage(arr))

[5, 2, 8, 11, 8, 4] sai: True  đúng: False
[1, 2, 3, 4, 5, 6, 7, 8] sai: True  đúng: True


**Vì sao sai:** khi `k` tăng dần, `dp[k−1]` đã chứa phần tử `x` vừa thêm ở cùng vòng → `dp[k]` có thể dùng `x` **hai lần**. Ví dụ `[5, 2, 8, 11, 8, 4]`: `S = 38, n = 6`, với `k = 3` cần tập 3 phần tử tổng `19`. Không có tập như vậy, nhưng khi xử lý `x = 4`, code sai lần lượt tạo `4 ∈ dp[1]`, rồi `4 + 4 = 8 ∈ dp[2]`... và `11 + 4 + 4 = 19 ∈ dp[3]` — dùng số 4 hai lần → trả `True` sai.

**Code đúng:** duyệt `k` **giảm dần** (như knapsack 0/1).

### Lỗi 3 — Meet in the middle quên loại trường hợp "cả mảng"

**Code sai:**

In [22]:
def wrong_mitm_full(nums):
    n, S = len(nums), sum(nums)
    if n == 1:
        return False
    a = [n * x - S for x in nums]
    L, R = a[: n // 2], a[n // 2:]
    def gen(arr):
        sums = [0]
        for x in arr:
            sums += [s + x for s in sums]
        return sums
    sl, sr = gen(L), gen(R)
    right = set(sr[1:])
    if 0 in right or 0 in sl[1:]:
        return True
    return any(-s in right for s in sl[1:])      # SAI: gồm cả toàn bộ L

print("sai:", wrong_mitm_full([3, 1]), " đúng:", Solution().splitArraySameAverage([3, 1]))

sai: True  đúng: False


**Vì sao sai:** tổng toàn bộ `a` luôn bằng 0. Ghép "toàn bộ `L`" với "toàn bộ `R`" luôn cho tổng 0 → luôn `True`. Nhưng đó là chọn `A` = cả mảng, `B` rỗng — không hợp lệ.

**Code đúng:** khi ghép, bỏ mask "toàn bộ `L`" (cách 4).

### Lỗi 4 — Quên trường hợp `n = 1`

Với `n = 1` không thể chia thành hai mảng khác rỗng. Cách bitset tự xử lý (`half = 0` → không có `k`), nhưng cách meet in the middle cần `if n == 1: return False` vì `L` rỗng.

## 14. Mẹo & nhận dạng pattern

- **"Hai phần có trung bình bằng nhau"** ⇔ **"một phần có trung bình bằng cả mảng"**. Tương tự: "hai phần có tổng bằng nhau" ⇔ "một phần có tổng bằng S/2".
- **Trừ đi trung bình** (`aᵢ = n·xᵢ − S`) biến "trung bình bằng" thành "tổng bằng 0" — loại bỏ ràng buộc số phần tử.
- **n ≤ 30–40** → nghĩ ngay tới **meet in the middle**.
- **Tổng nhỏ (≤ vài trăm nghìn)** → **DP tổng tập con**, tăng tốc bằng **bitset** `B |= B << x`.
- DP 0/1 theo số phần tử: duyệt `k` **giảm dần**.
- Cùng họ: **416 Partition Equal Subset Sum**, **1755 Closest Subsequence Sum** (meet in the middle), **2035 Partition Array Into Two Arrays to Minimize Sum Difference**.

## 15. Biến thể

1. **Trả về phép chia cụ thể:** lưu vết DP hoặc lưu mask trong meet in the middle.
2. **Chia thành `m` phần cùng trung bình:** mỗi phần phải có trung bình `S/n`; bài trở thành phân hoạch nhiều tập — khó hơn nhiều (NP-khó tổng quát).
3. **Chia thành hai phần có kích thước bằng nhau và tổng bằng nhau** (n chẵn): chỉ cần `k = n/2`, tổng `S/2`.
4. **Có số âm:** cách 4 vẫn chạy nguyên; cách bitset cần dịch toàn bộ tổng lên (offset).
5. **Đếm số cách chia:** thay `set` bằng `Counter` (đếm theo `(k, tổng)`), chú ý chia đôi do đối xứng `A ↔ B`.

Minh hoạ biến thể 1 — trả về `A` cụ thể bằng meet in the middle:

In [23]:
def find_split(nums):
    n, S = len(nums), sum(nums)
    if n == 1:
        return None
    a = [n * x - S for x in nums]
    h = n // 2
    L, R = a[:h], a[h:]
    def gen(arr):
        sums = [0]
        for x in arr:
            sums += [s + x for s in sums]
        return sums
    sl, sr = gen(L), gen(R)
    pos_r = {}
    for m in range(1, len(sr)):
        pos_r.setdefault(sr[m], m)
    def build(mL, mR):
        idx = [i for i in range(h) if mL >> i & 1] + [h + j for j in range(n - h) if mR >> j & 1]
        return [nums[i] for i in idx]
    for m in range(1, len(sl)):
        if sl[m] == 0:
            return build(m, 0)
    if 0 in pos_r:
        return build(0, pos_r[0])
    for m in range(1, len(sl) - 1):
        if -sl[m] in pos_r:
            return build(m, pos_r[-sl[m]])
    return None

for arr in ([1, 2, 3, 4, 5, 6, 7, 8], [3, 1], [2, 0, 5, 6, 16, 12, 15, 12, 4]):
    A = find_split(arr)
    print(arr, "->", A, "| avg A =", None if A is None else sum(A) / len(A), "| avg cả mảng =", sum(arr) / len(arr))

[1, 2, 3, 4, 5, 6, 7, 8] -> [1, 8] | avg A = 4.5 | avg cả mảng = 4.5
[3, 1] -> None | avg A = None | avg cả mảng = 2.0
[2, 0, 5, 6, 16, 12, 15, 12, 4] -> [12, 4] | avg A = 8.0 | avg cả mảng = 8.0


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | thử mọi mask | `O(2ⁿ n)` | `O(1)` | `n ≤ 16`, đối chiếu |
| 2. DP set | `dp[k]` = tập tổng | `O(n² S)` | `O(n S)` | dễ hiểu, đủ với dữ liệu thường |
| 3. DP bitset | `dp[k] |= dp[k−1] << x` | `O(n² S/64)` | `O(n S/64)` | **nộp bài**, nhanh khi `S` nhỏ |
| 4. Meet in the middle | `aᵢ = n·xᵢ − S`, tổng 0 | `O(2^{n/2} n)` | `O(2^{n/2})` | khi giá trị lớn / có số âm |

## 17. Bài tập

**Bài 1 (đoán output).** `splitArraySameAverage([4, 4, 4, 4])`, `([1, 3])`, `([2, 4, 6])` = ?

**Bài 2 (tính tay).** Với `nums = [5, 3, 11, 19, 2]`: tính `S`, `n`, các `k ≤ n/2` hợp lệ và tổng cần. Có tách được không?

**Bài 3 (điền TODO).** Hoàn thiện `partition_equal_sum(nums)` (LeetCode 416): chia thành hai phần **tổng** bằng nhau, dùng bitset.

**Bài 4 (tự viết lại).** Viết lại cách 2 với **dừng sớm**: kiểm tra mục tiêu ngay sau khi thêm mỗi phần tử.

**Bài 5 (mini challenge).** Đếm số cách chọn tập con `A` (khác rỗng, khác cả mảng) sao cho `avg(A) = avg(nums)`. Dùng meet in the middle với `Counter`.

In [24]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def partition_equal_sum(nums):
    # TODO: S lẻ -> False; B = 1; B |= B << x; kiểm tra bit S//2
    pass

# Bài 4 — TODO
def split_dp_early(nums):
    # TODO: như split_dp_set nhưng sau mỗi x kiểm tra các (k, tổng cần)
    pass

# Bài 5 — TODO
def count_same_avg_subsets(nums):
    # TODO: Counter tổng (theo a = n*x - S) của mọi tập con mỗi nửa; ghép số cách có tổng 0; trừ rỗng và cả mảng
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: mọi phần tử bằng nhau → mọi tập con có cùng trung bình.
- Mức 2: `[1, 3]`: `k = 1`, cần tổng 2 → không.
- Mức 3: `[2, 4, 6]`: `S = 12, n = 3`, `k = 1` cần 4 → có `{4}` → `True`.

**Bài 2.**
- Mức 1: `S = 40`, `n = 5`, `S/n = 8`.
- Mức 2: `k = 1` cần 8 (không có); `k = 2` cần 16: `5 + 11` ✔.
- Mức 3: `True`, `A = [5, 11]`, `B = [3, 19, 2]` (trung bình đều 8).

**Bài 3.**
- Mức 1: tổng lẻ thì không thể.
- Mức 2: bitset tổng tập con.
- Mức 3: kiểm tra `B >> (S // 2) & 1`.

**Bài 4.**
- Mức 1: tính `targets` trước.
- Mức 2: sau vòng `for k`, duyệt `targets`.
- Mức 3: trả `True` ngay khi gặp.

**Bài 5.**
- Mức 1: `aᵢ = n·xᵢ − S`; đếm tập con có tổng 0.
- Mức 2: `Counter` cho mỗi nửa (kể cả rỗng); số cách = `Σ cntL[s] · cntR[−s]`.
- Mức 3: trừ 2 (tập rỗng và cả mảng, cả hai có tổng 0).

## 19. Đáp án tham khảo

In [25]:
S_ = Solution()
# Bài 1
assert S_.splitArraySameAverage([4, 4, 4, 4]) is True
assert S_.splitArraySameAverage([1, 3]) is False
assert S_.splitArraySameAverage([2, 4, 6]) is True

# Bài 2
assert sum([5, 3, 11, 19, 2]) == 40
assert S_.splitArraySameAverage([5, 3, 11, 19, 2]) is True

# Bài 3
def partition_equal_sum_ans(nums):
    S = sum(nums)
    if S % 2:
        return False
    B = 1
    for x in nums:
        B |= B << x
    return bool(B >> (S // 2) & 1)

assert partition_equal_sum_ans([1, 5, 11, 5]) is True
assert partition_equal_sum_ans([1, 2, 3, 5]) is False
for seed in range(300):
    r = random.Random(seed)
    a = [r.randint(1, 20) for _ in range(r.randint(1, 10))]
    brute = any(2 * sum(c) == sum(a) for k in range(len(a) + 1) for c in combinations(a, k))
    assert partition_equal_sum_ans(a) == brute

# Bài 4
def split_dp_early_ans(nums):
    n, S = len(nums), sum(nums)
    half = n // 2
    targets = {k: S * k // n for k in range(1, half + 1) if S * k % n == 0}
    if not targets:
        return False
    dp = [set() for _ in range(half + 1)]
    dp[0].add(0)
    for x in nums:
        for k in range(half, 0, -1):
            if dp[k - 1]:
                dp[k] |= {s + x for s in dp[k - 1]}
        if any(t in dp[k] for k, t in targets.items()):
            return True
    return False

for seed in range(500):
    r = random.Random(seed)
    a = [r.randint(0, 50) for _ in range(r.randint(1, 12))]
    assert split_dp_early_ans(a) == split_brute(a)

# Bài 5
from collections import Counter
def count_same_avg_subsets_ans(nums):
    n, S = len(nums), sum(nums)
    a = [n * x - S for x in nums]
    def cnt(arr):
        sums = [0]
        for x in arr:
            sums += [s + x for s in sums]
        return Counter(sums)
    cl, cr = cnt(a[: n // 2]), cnt(a[n // 2:])
    total = sum(c * cr[-s] for s, c in cl.items())
    return total - 2                      # bỏ tập rỗng và cả mảng

def count_brute(nums):
    n, S = len(nums), sum(nums)
    return sum(1 for mask in range(1, (1 << n) - 1)
               if n * sum(nums[i] for i in range(n) if mask >> i & 1) == bin(mask).count('1') * S)

for seed in range(300):
    r = random.Random(seed)
    a = [r.randint(0, 10) for _ in range(r.randint(1, 11))]
    assert count_same_avg_subsets_ans(a) == count_brute(a), a
print("count([1..8]) =", count_same_avg_subsets_ans([1, 2, 3, 4, 5, 6, 7, 8]))
print("Tất cả đáp án đúng")

count([1..8]) = 16
Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (bitset):**

```python
n, S = len(nums), sum(nums); half = n // 2
targets = [k for k in range(1, half + 1) if S * k % n == 0]
if not targets: return False
dp = [0] * (half + 1); dp[0] = 1
for x in nums:
    for k in range(half, 0, -1):
        dp[k] |= dp[k - 1] << x
return any(dp[k] >> (S * k // n) & 1 for k in targets)
```

**Template (meet in the middle, tổng 0):** `a = [n*x - S]`, sinh tổng hai nửa, kiểm tra 0 ở mỗi nửa, ghép `sL` (khác rỗng, khác toàn bộ L) với `−sL` trong nửa phải.

**Checklist trước khi submit:**

- [ ] So sánh bằng **số nguyên** (`n·sA == k·S`), không dùng số thực.
- [ ] Chỉ xét `1 ≤ k ≤ n/2`, cắt tỉa bằng `S·k % n == 0`.
- [ ] DP duyệt `k` **giảm dần**.
- [ ] Meet in the middle: loại trường hợp `A` = cả mảng.
- [ ] `n = 1` → `False`.